# A4 – Hybrid Search / RRF Fusion

- **Adapted from:** `all_rag_techniques/fusion_retrieval.ipynb`
- **Experiment ID:** `A4_HYBRID`
- **Corpus:** `report_data/raw`
- **Evaluation set:** `report_data/evaluation/questions.json`
- **Purpose:** Compare dense-only, BM25-only, and hybrid (RRF fusion) retrieval. Uses best chunking from A1/A2 and best query strategy from A3. Only retrieval method changes.

## Hypothesis

- Dense retrieval handles semantic queries well but misses exact identifiers.
- BM25 captures keywords and identifiers but misses paraphrased content.
- Hybrid (RRF) should improve Recall@K and nDCG across both query types.

## Control Variables

- Chunking: best from A1/A2
- Query: best strategy from A3 (or original)
- Top-K candidates: same budget for fair comparison
- Prompt: same naive prompt
- LLM: same

## 1. Setup

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from report_common.config import load_config, print_config_summary, save_config_snapshot
from report_common.models import build_llm, build_embeddings
from report_common.evaluation import compute_retrieval_metrics
from report_common.io import save_jsonl, save_csv_summary, build_result_record, Timer

In [ ]:
config = load_config()
print_config_summary(config)

EXPERIMENT_ID = "A4_HYBRID"
NOTEBOOK = "05_fusion_retrieval.ipynb"
SEED = config["seed"]
CONFIG_HASH = config["_config_hash"]
TOP_K = config["baseline"]["top_k"]
RRF_K = config["retrieval"]["rrf_k"]
CANDIDATE_TOP_K = config["retrieval"]["candidate_top_k"]

In [ ]:
llm = build_llm(config)
embeddings = build_embeddings(config)
print(f"LLM OK: {llm.invoke('hi').content[:30]}")
print(f"Embedding OK: dim={len(embeddings.embed_query('test'))}")

## 2. Load & Chunk Corpus

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

raw_data_path = PROJECT_ROOT / config["paths"]["raw_data"]
documents = []
for pdf_file in raw_data_path.glob("*.pdf"):
    documents.extend(PyPDFLoader(str(pdf_file)).load())

# Best chunk config from A1/A2 (UPDATE after running)
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(documents)
for c in chunks:
    c.page_content = c.page_content.replace('\t', ' ')

print(f"Chunks: {len(chunks)} (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

# Load questions
with open(PROJECT_ROOT / config["paths"]["questions"], "r", encoding="utf-8") as f:
    eval_questions = json.load(f)
print(f"Questions: {len(eval_questions)}")

## 3. Build Dense Index

In [ ]:
with Timer() as t_dense:
    vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"Dense index: {vectorstore.index.ntotal} vectors ({t_dense.elapsed:.2f}s)")

## 4. Build BM25 Index

In [ ]:
# Tokenize chunks for BM25
chunk_texts = [c.page_content for c in chunks]
tokenized_chunks = [text.lower().split() for text in chunk_texts]

with Timer() as t_bm25:
    bm25 = BM25Okapi(tokenized_chunks)
print(f"BM25 index built ({t_bm25.elapsed:.2f}s)")

## 5. Define Retrieval Strategies

In [ ]:
def retrieve_dense(query: str, k: int = TOP_K):
    """Dense vector retrieval."""
    docs = vectorstore.similarity_search_with_score(query, k=k)
    return [(chunks[chunk_texts.index(d.page_content)] if d.page_content in chunk_texts else d, score)
            for d, score in docs]


def retrieve_bm25(query: str, k: int = TOP_K):
    """BM25 sparse retrieval."""
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:k]
    return [(chunks[i], scores[i]) for i in top_indices]


def retrieve_hybrid_rrf(query: str, k: int = TOP_K, rrf_k: int = RRF_K):
    """Hybrid retrieval with Reciprocal Rank Fusion."""
    # Get candidates from both
    dense_results = retrieve_dense(query, k=CANDIDATE_TOP_K)
    bm25_results = retrieve_bm25(query, k=CANDIDATE_TOP_K)

    # Build RRF scores
    rrf_scores = {}  # chunk_content -> (score, chunk)

    for rank, (doc, _) in enumerate(dense_results, start=1):
        key = doc.page_content[:100]  # Use prefix as key for dedup
        if key not in rrf_scores:
            rrf_scores[key] = {"score": 0.0, "doc": doc, "dense_rank": rank, "bm25_rank": None}
        rrf_scores[key]["score"] += 1.0 / (rrf_k + rank)
        rrf_scores[key]["dense_rank"] = rank

    for rank, (doc, _) in enumerate(bm25_results, start=1):
        key = doc.page_content[:100]
        if key not in rrf_scores:
            rrf_scores[key] = {"score": 0.0, "doc": doc, "dense_rank": None, "bm25_rank": rank}
        rrf_scores[key]["score"] += 1.0 / (rrf_k + rank)
        rrf_scores[key]["bm25_rank"] = rank

    # Sort by RRF score and return top-k
    sorted_results = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
    return [(item["doc"], item["score"]) for item in sorted_results[:k]]


RETRIEVAL_STRATEGIES = {
    "DENSE": retrieve_dense,
    "BM25": retrieve_bm25,
    "HYBRID_RRF": retrieve_hybrid_rrf,
}
print(f"Strategies: {list(RETRIEVAL_STRATEGIES.keys())}")

## 6. Run Evaluation

In [ ]:
NAIVE_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}
Answer:"""
)
answer_chain = NAIVE_PROMPT | llm

all_results = []

for strategy_name, retrieve_fn in RETRIEVAL_STRATEGIES.items():
    print(f"\n{'='*60}")
    print(f"RETRIEVAL: {strategy_name}")
    print(f"{'='*60}")

    for q in eval_questions:
        question = q["question"]
        relevant_docs = q.get("relevant_documents", [])

        with Timer() as t_ret:
            results_with_score = retrieve_fn(question, k=TOP_K)

        docs = [doc for doc, _ in results_with_score]
        scores = [score for _, score in results_with_score]
        retrieved_ids = [d.metadata.get("source", f"chunk_{i}") for i, d in enumerate(docs)]
        context = "\n\n".join([d.page_content for d in docs])

        with Timer() as t_gen:
            response = answer_chain.invoke({"context": context, "question": question})

        metrics = compute_retrieval_metrics(retrieved_ids, relevant_docs, k=TOP_K)

        record = build_result_record(
            experiment_id=f"{EXPERIMENT_ID}_{strategy_name}",
            notebook=NOTEBOOK,
            config_hash=CONFIG_HASH,
            seed=SEED,
            question_id=q["question_id"],
            question=question,
            answer=response.content,
            retrieved_documents=[
                {"content": d.page_content[:200], "score": float(s), "metadata": d.metadata}
                for d, s in results_with_score
            ],
            latency={
                "retrieval_seconds": t_ret.elapsed,
                "generation_seconds": t_gen.elapsed,
                "total_seconds": t_ret.elapsed + t_gen.elapsed,
            },
            usage={"context_chars": len(context)},
            metrics=metrics,
            retrieval_method=strategy_name,
        )
        all_results.append(record)

    print(f"  Done ({len(eval_questions)} questions)")

print(f"\nTotal: {len(all_results)} records")

## 7. Comparison Table

In [ ]:
summary_rows = []
for strategy_name in RETRIEVAL_STRATEGIES:
    strat_results = [r for r in all_results if r["retrieval_method"] == strategy_name]
    metrics_agg = {}
    for key in strat_results[0]["metrics"]:
        vals = [r["metrics"][key] for r in strat_results if r["metrics"].get(key) is not None]
        metrics_agg[key] = round(np.mean(vals), 3) if vals else None

    avg_latency = np.mean([r["latency"]["total_seconds"] for r in strat_results])
    summary_rows.append({
        "retrieval": strategy_name,
        "avg_latency_s": round(avg_latency, 3),
        **metrics_agg,
    })

df = pd.DataFrame(summary_rows)
print(df.to_string(index=False))

## 8. Save Results

In [ ]:
output_dir = PROJECT_ROOT / config["paths"]["results"]
save_jsonl(all_results, output_dir / "A4_fusion.jsonl")
save_csv_summary(summary_rows, output_dir / "A4_fusion_summary.csv")
save_config_snapshot(config, output_dir)

## 9. Observations

- Dense: best for semantic queries, struggles with `___`.
- BM25: best for exact terms, misses `___`.
- Hybrid RRF (k=60): combines both signals, achieves Recall@K=`___`, nDCG=`___`.
- RRF adds minimal latency over dense-only.
- Hybrid will be used as the retrieval method for A5+ (reranking stage).

_Fill in after running._